In [25]:
# CSV -> JSONL with cleaning and deduplication
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ---- configure paths ----
CSV_PATH = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/TREC_06.csv")
OUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/TREC_06.cleaned.jsonl")

# Optional: list columns to keep (None keeps all)
KEEP_COLUMNS = None  # e.g., ["label", "content", "subject"]

# Columns to drop if present
DROP_COLUMNS = ["sender", "receiver", "date"]

# Optional: define text columns that should be cleaned aggressively
TEXT_COLUMNS = ["content", "subject", "sender", "body"]

# Optional: define columns used for duplicate detection (None uses all columns)
DEDUP_COLUMNS = None  # e.g., ["content", "subject"]

# Subjects to drop (exact match, case-insensitive after cleaning)
SUBJECT_BLACKLIST = ["=?ISO-8859-1?Q?Compte_Suspect=E9_=21=21=21?="]

# Subject regex patterns to drop (case-insensitive)
SUBJECT_BLACKLIST_REGEX = [r"=\?.*\?.*\?.*\?="]

# Drop extremely short emails (character count after cleaning)
MIN_EMAIL_LENGTH = 80

# Columns to use for exact email text duplicate removal (first existing is used)
EMAIL_TEXT_COLUMNS = ["body", "content"]

# Drop emails containing non-ASCII characters (simple non-English heuristic)
DROP_NON_ASCII = True

# ---- helpers ----
_WHITESPACE_RE = re.compile(r"\s+")
_CONTROL_RE = re.compile(r"[\x00-\x1F\x7F]")
_ZERO_WIDTH_RE = re.compile(r"[\u200B\u200C\u200D\uFEFF]")
_NON_ASCII_RE = re.compile(r"[^\x00-\x7F]")


def clean_text(value: object) -> str | None:
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    text = str(value)
    # Normalize literal escape sequences and actual tab chars
    text = text.replace("\\t", " ").replace("\t", " ")
    # Remove zero-width and BOM characters that split words
    text = _ZERO_WIDTH_RE.sub("", text)
    text = _CONTROL_RE.sub(" ", text)
    text = text.replace("\u00a0", " ")
    text = _WHITESPACE_RE.sub(" ", text).strip()
    if not text:
        return None
    return text


def normalize_row(row: pd.Series, text_columns: list[str]) -> dict:
    data = row.to_dict()
    for col in text_columns:
        if col in data:
            data[col] = clean_text(data[col])
    # Drop empty fields
    return {k: v for k, v in data.items() if v is not None}


# ---- load ----
df = pd.read_csv(CSV_PATH)

# Drop unwanted columns if present
drop_cols = [c for c in DROP_COLUMNS if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

# Keep only selected columns if provided
if KEEP_COLUMNS is not None:
    df = df[[c for c in KEEP_COLUMNS if c in df.columns]]

# Ensure text columns list only includes existing columns
text_cols = [c for c in TEXT_COLUMNS if c in df.columns]

# Normalize text fields
df = df.copy()
for col in text_cols:
    df[col] = df[col].map(clean_text)

# Drop rows that are entirely empty after cleaning
df = df.dropna(how="all")

# Drop rows missing subject or body
if "subject" in df.columns and "body" in df.columns:
    df = df[df["subject"].notna() & df["body"].notna() & (df["subject"] != "") & (df["body"] != "")]

# Drop rows with blacklisted subject values or patterns
if "subject" in df.columns:
    if SUBJECT_BLACKLIST:
        blacklist = {clean_text(s).lower() for s in SUBJECT_BLACKLIST if s}
        df = df[~df["subject"].fillna("").str.lower().isin(blacklist)]
    if SUBJECT_BLACKLIST_REGEX:
        regex = re.compile("|".join(SUBJECT_BLACKLIST_REGEX), re.IGNORECASE)
        df = df[~df["subject"].fillna("").str.contains(regex)]

# Drop extremely short emails based on the first available text column
text_col = next((c for c in EMAIL_TEXT_COLUMNS if c in df.columns), None)
if text_col is not None:
    df = df[df[text_col].fillna("").str.len() >= MIN_EMAIL_LENGTH]

# Drop emails with non-ASCII characters in the main text
if DROP_NON_ASCII and text_col is not None:
    df = df[~df[text_col].fillna("").str.contains(_NON_ASCII_RE)]

# Deduplicate
if DEDUP_COLUMNS is None:
    df = df.drop_duplicates()
else:
    existing = [c for c in DEDUP_COLUMNS if c in df.columns]
    if existing:
        df = df.drop_duplicates(subset=existing)
    else:
        df = df.drop_duplicates()

# Remove exact duplicate email texts
if text_col is not None:
    df = df.drop_duplicates(subset=[text_col])

# Convert to records with per-row normalization
records = [normalize_row(row, text_cols) for _, row in df.iterrows()]

# Filter out any fully empty objects (if any slipped through)
records = [r for r in records if r]

# ---- write JSONL ----
OUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
with OUT_JSONL.open("w", encoding="utf-8") as f:
    for obj in records:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"Wrote {len(records)} records to {OUT_JSONL}")

# =========================================================
# Embedding-based near-duplicate removal
# =========================================================

INPUT_FILE = OUT_JSONL
OUTPUT_FILE = OUT_JSONL.with_suffix(".dedup.jsonl")
SIMILARITY_THRESHOLD = 0.75
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def extract_text(record: dict) -> str:
    if "body" in record and record["body"]:
        return record["body"]
    if "content" in record and record["content"]:
        return record["content"]
    if "subject" in record and record["subject"]:
        return record["subject"]
    return ""


def clean_email(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[a-f0-9]{16,}", " ", text)
    text = re.sub(r"\d{6,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


records = []
with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded {len(records)} emails for embedding")

if len(records) <= 1:
    cleaned_records = records
    with OUTPUT_FILE.open("w", encoding="utf-8") as f:
        for record in cleaned_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved cleaned dataset to: {OUTPUT_FILE}")
else:
    texts = [clean_email(extract_text(r)) for r in records]
    print("Loading embedding model...")
    embedder = SentenceTransformer(EMBED_MODEL)
    print("Embedding model loaded")
    print("Generating embeddings...")
    embeddings = embedder.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
)
    print(f"Embeddings shape: {embeddings.shape}")
    print("Finding near duplicates...")
    keep_indices: list[int] = []
    removed_indices: set[int] = set()
    for i in tqdm(range(len(records))):
        if i in removed_indices:
            continue
        keep_indices.append(i)
        tail = embeddings[i + 1 :]
        if tail.shape[0] == 0:
            continue
        current_embedding = embeddings[i].reshape(1, -1)
        similarities = cosine_similarity(current_embedding, tail)[0]
        for offset, sim in enumerate(similarities):
            j = i + 1 + offset
            if sim >= SIMILARITY_THRESHOLD:
                removed_indices.add(j)
    print(f"Removed {len(removed_indices)} near duplicates")
    cleaned_records = [records[i] for i in keep_indices]
    print(f"Final dataset size: {len(cleaned_records)}")
    with OUTPUT_FILE.open("w", encoding="utf-8") as f:
        for record in cleaned_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved cleaned dataset to: {OUTPUT_FILE}")

ParserError: Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.


In [26]:
# Clean full_dataset.json -> JSONL with body field
import json
from pathlib import Path

INPUT_JSON = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/full_dataset.json")
OUTPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/full_dataset.cleaned.jsonl")

with INPUT_JSON.open("r", encoding="utf-8") as f:
    data = json.load(f)

cleaned = []
for obj in data:
    if not isinstance(obj, dict):
        continue
    # Copy and remove unwanted keys
    item = dict(obj)
    item.pop("id", None)
    item.pop("body", None)
    item.pop("category_id", None)
    # Rename text -> body
    if "text" in item:
        item["body"] = item.pop("text")
    cleaned.append(item)

with OUTPUT_JSONL.open("w", encoding="utf-8") as f:
    for obj in cleaned:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"Wrote {len(cleaned)} records to {OUTPUT_JSONL}")

Wrote 13477 records to /home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/full_dataset.cleaned.jsonl
